# Machine Learning Pipeline - Model Training

La reproducibilidad es un pilar fundamental en todo proyecto de ciencia de datos y entrenamiento de modelos machine learning. Fijar la semilla aleatoria (`random_state` o `np.random.seed`) permite que los resultados sean consistentes a lo largo de distintos corridas y equipos, facilitando la validación y colaboración del código.

---

In [ ]:
# 2. IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os

# Para reproducibilidad
def set_seed(seed=0):
    import random
    np.random.seed(seed)
    random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass  # no torch installed
set_seed(0)

from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
from sklearn.metrics import accuracy_score, f1_score, classification_report

pd.set_option('display.max_columns', None)

---

## 3. CARGA DE DATOS TRANSFORMADOS

Asumimos que los datos transformados por el pipeline ya están almacenados en archivos CSV listos para ser cargados. Si el pipeline usa transformaciones no serializables, puede ser necesario ejecutar el pipeline previo sobre los datos originales; aquí optamos por leer los archivos generados por los notebooks anteriores.

In [ ]:
# Paths habituales (ajustar si es necesario)
PATH = 'proyecto_pipeline_machine_learning_muestra/'
X_train = pd.read_csv(os.path.join(PATH, 'X_train_transformed.csv'), index_col=0)
X_test = pd.read_csv(os.path.join(PATH, 'X_test_transformed.csv'), index_col=0)
y_train = pd.read_csv(os.path.join(PATH, 'y_train.csv'), index_col=0).squeeze()
y_test = pd.read_csv(os.path.join(PATH, 'y_test.csv'), index_col=0).squeeze()

# Determinar tipo de problema automáticamente
def is_regression(y):
    unique = np.unique(y)
    if np.issubdtype(y.dtype, np.floating):
        # N de valores únicos << N muestra suele ser regresión
        return len(unique) > 20
    return False

problem_type = 'regression' if is_regression(y_train) else 'classification'
print(f"Problema detectado: {problem_type}")

---

## 4. CARGA DE FEATURES SELECCIONADAS

In [ ]:
# Cargar features seleccionadas por el análisis anterior
features = pd.read_csv(os.path.join(PATH, 'selected_features.csv'))['0'].to_list()
X_train = X_train[features]
X_test = X_test[features]
print(f"N° features usadas: {len(features)}")

---

## 5. ENTRENAMIENTO DEL MODELO FINAL

In [ ]:
if problem_type == 'regression':
    model = Lasso(alpha=0.001, random_state=0, max_iter=5000)
else:
    model = LogisticRegression(penalty='l1', solver='saga', random_state=0, max_iter=1000)

model.fit(X_train, y_train)

---

## 6. EVALUACIÓN CON MÉTRICAS

In [ ]:
if problem_type == 'regression':
    # Predicciones (recordar Target estaba en log)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Revertimos el log para métricas interpretables
    y_train_exp = np.exp(y_train)
    y_test_exp = np.exp(y_test)
    y_pred_train_exp = np.exp(y_pred_train)
    y_pred_test_exp = np.exp(y_pred_test)
    
    train_mse = mean_squared_error(y_train_exp, y_pred_train_exp)
    train_rmse = root_mean_squared_error(y_train_exp, y_pred_train_exp)
    train_r2 = r2_score(y_train_exp, y_pred_train_exp)
    test_mse = mean_squared_error(y_test_exp, y_pred_test_exp)
    test_rmse = root_mean_squared_error(y_test_exp, y_pred_test_exp)
    test_r2 = r2_score(y_test_exp, y_pred_test_exp)
    
    print("EVALUACIÓN EN TRAIN:")
    print(f"MSE:  {train_mse:,.2f}")
    print(f"RMSE: {train_rmse:,.2f}")
    print(f"R2:   {train_r2:.4f}")
    print("\nEVALUACIÓN EN TEST:")
    print(f"MSE:  {test_mse:,.2f}")
    print(f"RMSE: {test_rmse:,.2f}")
    print(f"R2:   {test_r2:.4f}")
    print("\nPrecio promedio en test:", y_test_exp.mean())
else:
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    print("--- CLASIFICACIÓN TRAIN ---")
    print(f"Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
    print(f"F1: {f1_score(y_train, y_pred_train, average='weighted'):.4f}")
    print(classification_report(y_train, y_pred_train))
    print("--- CLASIFICACIÓN TEST ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
    print(f"F1: {f1_score(y_test, y_pred_test, average='weighted'):.4f}")
    print(classification_report(y_test, y_pred_test))

---

## 7. SCATTER: PREDICCIONES VS REALES

In [ ]:
plt.figure(figsize=(7,7))
if problem_type == 'regression':
    plt.scatter(np.exp(y_test), np.exp(model.predict(X_test)), alpha=0.6)
    plt.xlabel('Precio Real')
    plt.ylabel('Precio Predicho')
    plt.title('Predicción vs Valor Real - Test Set')
else:
    plt.scatter(y_test, model.predict(X_test), alpha=0.6)
    plt.xlabel('Clase Real')
    plt.ylabel('Clase Predicha')
    plt.title('Predicción vs Clase Real - Test Set')
plt.plot(plt.xlim(), plt.xlim(), '--r', label="Ideal")
plt.legend()
plt.grid()
plt.show()

---

## 8. DISTRIBUCIÓN DE ERRORES (Solo regresión)

In [ ]:
if problem_type == 'regression':
    errors = np.exp(y_test) - np.exp(model.predict(X_test))
    plt.figure(figsize=(8,4))
    plt.hist(errors, bins=40, color='steelblue', alpha=0.7)
    plt.xlabel('Error (Precio Real - Precio Predicho)')
    plt.ylabel('Frecuencia')
    plt.title('Distribución de Errores en el Test Set')
    plt.grid()
    plt.show()

> **Nota:** Una distribución de errores aproximadamente gaussiana centrada en cero indica que el modelo no tiene sesgo sistemático (ni subestima ni sobreestima consistentemente) y generaliza adecuadamente.

---

## 9. IMPORTANCIA DE FEATURES

In [ ]:
if problem_type == 'regression' or problem_type == 'classification':
    importances = pd.Series(np.abs(model.coef_.ravel()), index=features)
    importances = importances.sort_values(ascending=False)
    plt.figure(figsize=(18,6))
    importances.plot(kind='barh')
    plt.gca().invert_yaxis()
    plt.title("Importancia relativa de las features (|coeficiente|)")
    plt.xlabel('Magnitud absoluta del coeficiente')
    plt.tight_layout()
    plt.show()

---

## 10. GUARDADO DEL MODELO FINAL

In [ ]:
joblib.dump(model, os.path.join(PATH, 'modelo_final.joblib'))
print("Modelo guardado exitosamente en:", os.path.join(PATH, 'modelo_final.joblib'))